# **Data Profiling Overview**

This notebook performs a comprehensive data profiling process to evaluate the structure, quality, and characteristics of the dataset before any transformation or modeling.

### Objectives

* Identify schema details (column names, data types, nullability)
* Analyze data distribution and summary statistics
* Detect missing values, duplicates, and anomalies
* Validate data consistency (ranges, formats, constraints)
* Highlight potential data quality issues impacting downstream pipelines

### Scope of Analysis

* Row and column count
* Data types and schema validation
* Null / missing value assessment
* Distinct value counts and cardinality
* Statistical summary (mean, median, min, max, stddev for numerical fields)
* Pattern checks for categorical and date fields
* Outlier detection (basic threshold-based)

### Outcome

The output of this profiling step provides a clear understanding of the dataset’s health and readiness, enabling informed decisions for data cleaning, feature engineering, and pipeline design.

---

**Status:** Data profiling initiated...

**Author : Ritik**

Env : Devlopment / Testing

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Required Module are loaded successfully :)")   

Required Module are loaded successfully :)


In [7]:
spark = (
    SparkSession.builder
    .appName("DataProfiling")
    .getOrCreate()
)

print("spark Session build successfully :) ")

spark Session build successfully :) 


# **working with customers data**

In [8]:
try :
    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load("/workspaces/the_data_echo/row_data/raw_customers.csv")
    )

    print("DataFrame loaded successfully :) ")
except Exception as e :
    print(f"Error Occurred : {e}")

DataFrame loaded successfully :) 


In [33]:
df.sample(fraction=0.2, withReplacement=False).show(10, truncate=False)

+-----------+-----+----------+----------+--------------------+------+-----------------+---+-------------------------------+--------------+---------------------+-------------+----------+----------+----------+--------+-------------+-------+----------------+--------------+---------+--------------------+-----------------+-----------------+-----------------------+
|customer_id|title|first_name|last_name |full_name           |gender|date_of_birth    |age|email                          |phone         |address              |city         |state     |state_abbr|state_full|zip_code|country      |region |customer_segment|loyalty_points|is_active|account_created_date|preferred_channel|annual_income_usd|company                |
+-----------+-----+----------+----------+--------------------+------+-----------------+---+-------------------------------+--------------+---------------------+-------------+----------+----------+----------+--------+-------------+-------+----------------+--------------+------

## **Customer id data profiling** 

In [10]:
df.select("customer_id").filter(F.col("customer_id").isNull()).show()

+-----------+
|customer_id|
+-----------+
+-----------+



In [11]:
df.groupBy("customer_id").count().filter("count > 1").show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [12]:
df.select("customer_id").dtypes

[('customer_id', 'int')]

## **Customer title data profiling**

In [13]:
df.select("title").filter(F.col("title").isNotNull()).count()

282

In [14]:
df.select("title").distinct().show()

+-----+
|title|
+-----+
|Prof.|
|  Ms.|
|  Mr.|
| Mrs.|
|  Dr.|
| NULL|
+-----+



In [15]:
df.groupBy("title").count().filter("count > 1").show()

+-----+-----+
|title|count|
+-----+-----+
|Prof.|   57|
|  Ms.|   60|
| NULL|  358|
|  Mr.|   46|
| Mrs.|   71|
|  Dr.|   48|
+-----+-----+



## **Customer First Name data profiling**

In [16]:
df.select("first_name").filter(F.col("first_name").isNull()).count()

0

In [17]:
df.sample(fraction=0.02, withReplacement=False).select("first_name").filter(F.length(F.trim(F.col("first_name"))) >= 2).show()

+-----------+
| first_name|
+-----------+
|      Sarah|
|     Daniel|
|      Jason|
|     Samuel|
|  Christine|
|     Sharon|
|     Ashley|
|      JULIE|
|Christopher|
|    Jessica|
|       Paul|
|  Christine|
|    Carolyn|
|     Daniel|
|    DEBORAH|
+-----------+



In [18]:
df.withColumn("FirstName",
              F.col("first_name"))\
  .withColumn("FirstName2",
              F.trim(F.initcap(F.col("first_name")))).select("FirstName", "FirstName2")\
              .filter(F.col("FirstName") != F.col("FirstName2"))\
              .show()

+-----------+----------+
|  FirstName|FirstName2|
+-----------+----------+
|     rachel|    Rachel|
|     illiam|    Illiam|
|    cynthia|   Cynthia|
|   Anthony |   Anthony|
|   nicholas|  Nicholas|
|        AMY|       Amy|
|      Kaen |      Kaen|
|      maria|     Maria|
|   Shirley |   Shirley|
|   JONATHAN|  Jonathan|
|     Maria |     Maria|
|    shirley|   Shirley|
|  Margaret |  Margaret|
|      DONNA|     Donna|
|       jose|      Jose|
|     brenda|    Brenda|
|       Lisa|      Lisa|
|     David |     David|
|     nicole|    Nicole|
|      carol|     Carol|
+-----------+----------+
only showing top 20 rows


## **Customer Last Name data profiling**

In [19]:
df.select("last_name").filter(F.col("last_name").isNull()).show()

+---------+
|last_name|
+---------+
+---------+



In [20]:
df.sample(fraction=0.03, withReplacement=False).select("last_name").show(truncate=False)

+---------+
|last_name|
+---------+
|Watson   |
|Harris   |
|James    |
|gomez    |
|MURPHY   |
|Cruz     |
|mendoza  |
|Foster   |
|White    |
|Gonzalez |
|jackson  |
|Roberts  |
|moore    |
|williams |
|Walker   |
|lopez    |
|Collins  |
|Sanders  |
+---------+



In [21]:
df.withColumn("LastName",
              F.col("last_name"))\
  .withColumn("LastName2",
              F.trim(F.initcap(F.col("last_name")))).select("LastName", "LastName2")\
              .filter(F.col("LastName") != F.col("LastName2"))\
              .show()

+-----------+---------+
|   LastName|LastName2|
+-----------+---------+
|       king|     King|
|   Mendoza |  Mendoza|
|      MOORE|    Moore|
|  Peterson | Peterson|
|       KING|     King|
|      King |     King|
|  Castillo | Castillo|
|      REYES|    Reyes|
|      scott|    Scott|
|  Martinez | Martinez|
|      jones|    Jones|
|      Diaz |     Diaz|
|     Gomez |    Gomez|
|    Murphy |   Murphy|
|     MARTIN|   Martin|
|   Johnson |  Johnson|
|  Campbell | Campbell|
|  Anderson | Anderson|
|     martin|   Martin|
|   williams| Williams|
+-----------+---------+
only showing top 20 rows


## **Customer Full Name data profiling**

In [22]:
df.select("full_name").filter(F.col("full_name").isNull()).show()

+---------+
|full_name|
+---------+
+---------+



In [23]:
df.sample(fraction=0.03, withReplacement=False).select("full_name").show(truncate=False)

+--------------------+
|full_name           |
+--------------------+
|Dorothy Jackson     |
|Dr. Sarah Jones     |
|Maria Gomez         |
|Ronald Edwards      |
|Frank Clark         |
|Ashley Anderson     |
|Mrs. Daniel Nguyen  |
|Prof. Jessica Torres|
|Charles Murphy      |
|Ms. Shirley Phillips|
|Prof. Joyce Moore   |
|Mrs. Anthony Howard |
|Gary Castillo       |
|Donna King          |
|Ms. Nancy Nguyen    |
|Dr. Patrick Thomas  |
|Prof. Diane Moore   |
|Dr. Dorothy Chavez  |
|Andrew Ross         |
|Ms. Steven Turner   |
+--------------------+
only showing top 20 rows


In [24]:
df.withColumn("FullName",
              F.col("full_name"))\
  .withColumn("FullName2",
              F.trim(F.initcap(F.col("full_name")))).select("FullName", "FullName2")\
              .filter(F.col("FullName") != F.col("FullName2"))\
              .show()

+--------+---------+
|FullName|FullName2|
+--------+---------+
+--------+---------+



# **Customer Full Name data profiling**

In [25]:
df.select("gender").filter(F.col("gender").isNull()).count()

55

In [26]:
df.groupBy("gender").count().show()

+-----------------+-----+
|           gender|count|
+-----------------+-----+
|       non-binary|   47|
|                F|   37|
|Prefer not to say|   39|
|             NULL|   55|
|           Female|   57|
|           female|   41|
|                M|   45|
|            Other|   53|
|             MALE|   47|
|             male|   43|
|       Non-Binary|   42|
|             Male|   45|
|               NB|   48|
|           FEMALE|   41|
+-----------------+-----+



In [27]:
df.groupBy(F.trim(F.initcap(F.col("gender"))).alias("gender")).count().show()

+-----------------+-----+
|           gender|count|
+-----------------+-----+
|                F|   37|
|Prefer Not To Say|   39|
|             NULL|   55|
|           Female|  139|
|               Nb|   48|
|                M|   45|
|            Other|   53|
|       Non-binary|   89|
|             Male|  135|
+-----------------+-----+



## **Customer data of birth data profiling**

In [47]:
df.select("date_of_birth").filter(F.col("date_of_birth").isNull()).count()

0

In [48]:
df.sample(fraction=0.02, withReplacement=False).select("date_of_birth").show(truncate=False)

+--------------+
|date_of_birth |
+--------------+
|03/18/1984    |
|1963/04/23    |
|1997/02/16    |
|09/05/1993    |
|Feb 07, 2001  |
|15-06-1982    |
|04/17/1989    |
|1985-11-08    |
|May 31, 1997  |
|22/01/1981    |
|1988-04-23    |
|April 07, 1970|
|2002-12-22    |
|1983/04/30    |
+--------------+



In [49]:
date_pattern = df.withColumn(
    "pattern",
    F.regexp_replace(
        F.regexp_replace(F.trim(F.col("date_of_birth")), "[0-9]", "9"),
        "[A-Za-z]+", "A"
    )
)

In [50]:
date_pattern.groupBy("pattern").count().filter("count > 1").show()

+----------+-----+
|   pattern|count|
+----------+-----+
|A 99, 9999|  170|
|99-99-9999|   88|
|99/99/9999|  176|
|9999-99-99|   98|
|9999/99/99|  108|
+----------+-----+



In [64]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^[A-Za-z]+ \d{1,2}, \d{4}$")).count()

170

In [65]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^[A-Za-z]+ \d{1,2}, \d{4}$")).show(10, truncate=False)

+-----------------+
|date_of_birth    |
+-----------------+
|February 20, 2001|
|August 01, 1961  |
|Jan 23, 1987     |
|February 28, 1961|
|October 15, 1988 |
|Nov 25, 1985     |
|Feb 21, 1962     |
|December 01, 1963|
|June 05, 1989    |
|Jun 05, 1991     |
+-----------------+
only showing top 10 rows


In [ ]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^\d{1,2}-\d{1,2}-\d{4}$")).count()

88

In [66]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^\d{1,2}-\d{1,2}-\d{4}$")).show(10, truncate=False)

+-------------+
|date_of_birth|
+-------------+
|05-09-1968   |
|04-12-1969   |
|29-01-1982   |
|11-12-1971   |
|22-10-2002   |
|09-02-1994   |
|20-02-2002   |
|29-05-1991   |
|31-01-1967   |
|09-10-1982   |
+-------------+
only showing top 10 rows


In [63]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^\d{1,2}/\d{1,2}/\d{4}$")).count()

176

In [62]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^\d{1,2}/\d{1,2}/\d{4}$")).show(10, truncate=False)

+-------------+
|date_of_birth|
+-------------+
|22/03/1976   |
|04/05/1969   |
|25/11/1987   |
|13/04/1970   |
|02/03/1987   |
|04/11/2001   |
|04/22/1986   |
|10/11/1987   |
|03/18/1984   |
|03/23/1970   |
+-------------+
only showing top 10 rows


In [71]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^\d{4}-\d{1,2}-\d{1,2}$")).count()

98

In [72]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^\d{4}-\d{1,2}-\d{1,2}$")).show(10, truncate=False)

+-------------+
|date_of_birth|
+-------------+
|2000-02-08   |
|1967-10-18   |
|2000-02-02   |
|1997-12-31   |
|1973-06-03   |
|1996-03-15   |
|1980-04-04   |
|1962-03-30   |
|1982-09-23   |
|1992-08-15   |
+-------------+
only showing top 10 rows


In [74]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^\d{4}/\d{2}/\d{2}$")).count()

108

In [77]:
df.select("date_of_birth").filter(F.col("date_of_birth").rlike(r"^\d{4}/\d{2}/\d{2}$")).show(10, truncate=False)

+-------------+
|date_of_birth|
+-------------+
|1994/02/28   |
|1969/09/20   |
|2001/12/01   |
|1998/08/21   |
|1988/07/25   |
|1967/02/19   |
|1987/01/11   |
|1971/02/07   |
|1963/04/23   |
|1962/10/28   |
+-------------+
only showing top 10 rows


# **Customer age data profiling**

In [89]:
df.createOrReplaceTempView("customers")

In [79]:
df.select("age").filter(F.col("age").isNull()).count()

0

In [98]:
spark.sql("""
            SELECT 
                MAX(age)    as max_age,
                MIN(age)    as min_age,
                AVG(age)    as avg_age,
                MODE(age)   as mode_age,
                MEDIAN(age) as med_age
            FROM customers LIMIT 
""").show()

+-------+-------+---------+--------+-------+
|max_age|min_age|  avg_age|mode_age|med_age|
+-------+-------+---------+--------+-------+
|     65|     22|42.765625|      48|   42.0|
+-------+-------+---------+--------+-------+



In [88]:
df.select(F.max("age")).show()

+--------+
|max(age)|
+--------+
|      65|
+--------+



In [55]:
df.show(truncate=False)

+-----------+-----+----------+----------+------------------------+-----------------+-----------------+---+-------------------------------+--------------+---------------------------+----------------+----------+----------+----------+--------+-------------+---------+----------------+--------------+---------+--------------------+-----------------+-----------------+----------------------+
|customer_id|title|first_name|last_name |full_name               |gender           |date_of_birth    |age|email                          |phone         |address                    |city            |state     |state_abbr|state_full|zip_code|country      |region   |customer_segment|loyalty_points|is_active|account_created_date|preferred_channel|annual_income_usd|company               |
+-----------+-----+----------+----------+------------------------+-----------------+-----------------+---+-------------------------------+--------------+---------------------------+----------------+----------+----------+------